# Flow variation in a pump recording

FORGE | Exploratory analysis of SKAB experiment `valve1/1`

This analysis examines whether flow measurements distinguish normal from anomalous observations in a laboratory pump recording.

Observations labeled anomalous had a lower median flow, although their range overlapped with the normal group. A trailing rolling median showed when sustained changes occurred. The findings describe one development recording, with detector performance still to be evaluated.


## Dataset and provenance

The recording comes from the [Skoltech Anomaly Benchmark (SKAB)](https://github.com/waico/SKAB/blob/b2c0d46c2971dcbfe71e26087b6d231998bb91c2/README.md), developed by Iurii D. Katser and Vyacheslav O. Kozitsin. Experiment `valve1/1` belongs to the inlet-valve closure experiments in the [source catalog](https://github.com/waico/SKAB/blob/b2c0d46c2971dcbfe71e26087b6d231998bb91c2/data/README.md).

The sample contains 1,145 observations across 11 columns and spans 20 minutes. The [sample manifest](../data/sample-manifest.json) records its source revision, checksum, and local location. The dataset retains its source license, documented in the [data notes](../data/README.md).

Before analysis, the loader verifies the checksum and checks the required columns, timestamp order, numeric values, and annotations. The recording is reserved for development because its annotations have already been inspected. It is excluded from the final test set.


In [ ]:
from forge.data.sample import load_sample

df = load_sample()
print(f"Loaded sample recording: {df.shape}")

In [ ]:
print(df.head(5))
column_names = df.columns.to_list()
print(column_names)

## Measurements and reference annotations

The recording contains eight sensor channels, a timestamp, and two source annotation columns.

| Columns | Role |
|---|---|
| `Accelerometer1RMS`, `Accelerometer2RMS`, `Current`, `Pressure`, `Temperature`, `Thermocouple`, `Voltage`, `Volume Flow RateRMS` | Sensor measurements |
| `datetime` | Observation time |
| `anomaly` | Source label: 0 for normal observations, 1 for anomalous observations |
| `changepoint` | Source marker: 1 at a marked transition, 0 elsewhere |

These definitions follow the [SKAB documentation](https://github.com/waico/SKAB/blob/b2c0d46c2971dcbfe71e26087b6d231998bb91c2/README.md#datasets). An anomalous period can span many observations. A change point marks a transition, including a return to the normal label.

Source annotations provide reference labels for this comparison and are excluded from the sensor input table.


In [ ]:
# extend dataframe
sensor_columns = df.columns.difference(["datetime", "anomaly", "changepoint"])
label_columns = df.columns.intersection(["anomaly", "changepoint"])
print(f"Sensor columns: {sensor_columns.to_list()}")
print(f"Label columns: {label_columns.to_list()}")

In [ ]:
print(df["anomaly"].value_counts())

The sensor table `X` contains eight columns, while `labels` contains the two annotation columns. Both keep the original row alignment.

Column selection relies on the pinned sample's ordering, with the timestamp first and the annotations last. The current extraction code therefore depends on that schema.


In [ ]:
X = df[column_names[1 : len(column_names) - 2]]
labels = df[column_names[len(column_names) - 2 : len(column_names)]]
print(X)
print(labels)
print(f"X shape: {X.shape} | X column names: {X.columns.to_list()}")
print(f"Labels shape: {labels.shape} | Labels column names: {labels.columns.to_list()}")

## Data quality

The audit checks missing values in each column, duplicate timestamps, and elapsed time between consecutive observations. It uses the validated recording in its original order without resampling or filling gaps.

In [ ]:
missing_by_column = df.isna().sum()
duplicate_timestamps = df["datetime"].duplicated().sum()
sampling_intervals = df["datetime"].diff().dt.total_seconds()
interval_counts = sampling_intervals.value_counts().sort_index()

print("Missing values per column:")
print(missing_by_column.to_string())
print(f"Total missing values: {missing_by_column.sum()}")
print(f"Duplicate timestamps: {duplicate_timestamps}")
print("Sampling intervals (seconds):")
print(interval_counts.to_string())

### Complete values with irregular sampling

| Check | Result |
|---|---:|
| Missing values across all columns | 0 |
| Duplicate timestamps | 0 |
| One-second intervals | 1,088 |
| Two-second intervals | 56 |

The 1,145 observations yield 1,144 consecutive intervals. The first observation has no preceding interval and is excluded from the interval counts.

No recorded cells are empty, but the timestamps contain gaps relative to a one-second cadence. These gaps do not establish why measurements are absent. The analysis preserves the recorded timestamps and uses elapsed-time windows for temporal summaries.

## Flow comparison

Flow from `Volume Flow RateRMS`, measured in liters per minute (L/min), was grouped by the source `anomaly` label. Each group was summarized by its count, median, minimum, and maximum.


In [ ]:
# Select normal_flow and anomalous_flow from df.
normal_flow = df.loc[df["anomaly"] == 0, "Volume Flow RateRMS"]
anomalous_flow = df.loc[df["anomaly"] == 1, "Volume Flow RateRMS"]


print("Normal flow statistics:")
print(f"Count: {normal_flow.count()}")
print(f"Median: {normal_flow.median()}")
print(f"Min: {normal_flow.min()}")
print(f"Max: {normal_flow.max()}")

print("Anomalous flow statistics:")
print(f"Count: {anomalous_flow.count()}")
print(f"Median: {anomalous_flow.median()}")
print(f"Min: {anomalous_flow.min()}")
print(f"Max: {anomalous_flow.max()}")

### Lower median flow with overlapping ranges

| Source label | Observations | Median flow (L/min) | Observed range (L/min) |
|---|---:|---:|---:|
| Normal (`anomaly = 0`) | 743 | 32 | 31 to 33 |
| Anomalous (`anomaly = 1`) | 402 | 31 | 30 to 32 |

The anomalous group had a lower median, while the observed ranges overlapped between 31 and 32 L/min. A single instantaneous flow cutoff therefore cannot perfectly separate the source labels in this recording.


## Temporal variation

Flow measurements were indexed by `datetime` and summarized with a trailing 30-second rolling median. Each window includes the current observation and earlier observations within that period. At the start of the recording, the calculation uses the observations available so far.

The recording contains 56 two-second sampling intervals among otherwise one-second intervals. A window defined by elapsed time preserves these gaps without inserting measurements.


In [ ]:
import plotly.express as px

flow_by_time = df.set_index("datetime")["Volume Flow RateRMS"]
print(flow_by_time.head())
# Median of available observations in the preceding 30 seconds.
flow_trend = flow_by_time.rolling("30s").median()
print(flow_trend.head(10))

In [ ]:
fig = px.scatter(
    df,
    x="datetime",
    y="Volume Flow RateRMS",
    color="anomaly",
    title="Volume Flow RateRMS Over Time",
)
fig.add_scatter(
    x=flow_trend.index,
    y=flow_trend,
    mode="lines",
    name="30s Rolling Median",
    line=dict(color="black"),
)

Running the plotting cells produces a scatter plot of measured flow, colored by the source anomaly labels, with the trailing median in black. Chart outputs are generated locally, and the numerical summary remains in the report.

The smoothed series showed a sustained reduction around 10:45 and an increase around 10:51. The plot places the grouped statistics in time, showing when flow changed during the recording.


## Interpretation and limitations

The overlap between flow measurements motivates evaluation of temporal and multivariate features. The comparison also pools normal observations from before and after the anomalous period, which may conceal differences between those periods.

The rolling median makes sustained variation easier to see, but it can hide brief changes and respond later than individual measurements. A detection delay remains to be measured once a detector and alert rule have been defined.

The findings describe one laboratory experiment. Establishing a failure mechanism or performance across equipment and operating conditions requires further evidence. Evaluation will use separate experiments for training, validation, and testing, with transformations and thresholds fitted without access to the test set.
